# 01 · Visium HD — Data Loading
**Dataset**: GEO GSM9936742 / HD-02 · Human femoral head · Visium HD (56 µm spatial bins)

**Goal**: Load gene expression data, match spatial coordinates, inspect the tissue image, and save raw data as results/adata/01_raw.h5ad.

## Input Data
This notebook uses three files from GEO GSM9936742:

| Input file | Description |
|---|---|
| `GSM9936742_HD-02_filtered_feature_bc_matrix.h5` | Gene expression count matrix (HDF5) |
| `GSM9936742_HD-02_tissue_positions.parquet` | Spatial coordinates of each bin (Parquet) |
| `GSM9936742_HD-02_tissue_hires_image.png` | High-resolution tissue image (PNG) |

All files are located in the `data/` directory. The HDF5 and Parquet files are required; the PNG is used as a reference.

**Execution**: Run cells in order using **Run → Run All Cells**.

**Output**: results/adata/01_raw.h5ad

**Note**: Raw counts are preserved without QC filtering or normalization. Tissue-image registration is not validated in this step.

## 0 · Imports and project paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from PIL import Image
from scipy import sparse


def setup_paths(prefix="GSM9936742_HD-02"):
    """Find the project root and validate the three expected input files."""
    root = Path.cwd().resolve()
    if not (root / "data").is_dir() and (root.parent / "data").is_dir():
        root = root.parent
    if not (root / "data").is_dir():
        raise FileNotFoundError(
            "Cannot find data/. Start JupyterLab from the project root or notebooks/."
        )

    paths = {
        "H5": root / "data" / f"{prefix}_filtered_feature_bc_matrix.h5",
        "POSITIONS": root / "data" / f"{prefix}_tissue_positions.parquet",
        "IMAGE": root / "data" / f"{prefix}_tissue_hires_image.png",
        "OUTPUT": root / "results" / "adata" / "01_raw.h5ad",
    }
    for name in ("H5", "POSITIONS", "IMAGE"):
        if not paths[name].is_file():
            raise FileNotFoundError(f"Required input not found: {paths[name]}")
    return root, paths


PREFIX = "GSM9936742_HD-02"
ROOT, PATHS = setup_paths(PREFIX)
H5, POSITIONS, IMAGE, OUTPUT = (
    PATHS["H5"], PATHS["POSITIONS"], PATHS["IMAGE"], PATHS["OUTPUT"]
)

print("Python:", sys.executable)
print("Scanpy:", version("scanpy"))
print("Project root:", ROOT)
print("All three input files found.")

## 1 · Load the raw 10x count matrix

Keep the count matrix sparse and unchanged. Gene symbols can repeat; preserve original gene IDs in `adata.var["gene_ids"]`.

In [ ]:
def load_count_matrix(h5_path):
    """Read the original sparse 10x count matrix without normalization."""
    adata = sc.read_10x_h5(h5_path, gex_only=True)
    adata.var_names_make_unique()
    assert adata.obs_names.is_unique, "Count matrix barcodes are not unique"
    assert adata.var_names.is_unique, "Gene names must be unique"
    assert sparse.issparse(adata.X), "Expected a sparse count matrix"
    return adata


adata = load_count_matrix(H5)
print(adata)
print("Matrix shape (bins × genes):", adata.shape)
print("Nonzero count entries:", adata.X.nnz)
print("Sparse format:", adata.X.getformat())
print("First barcodes:", adata.obs_names[:3].tolist())

## 2 · Read and align spatial coordinates

Match by barcode **rather than row order**. `obsm["spatial"]` stores **(x, y) = (pixel column, pixel row)** in **full-resolution image coordinates**.

In [ ]:
def align_spatial_positions(adata, positions_path):
    """Match coordinates by barcode, preserving the matrix barcode order."""
    pos = pd.read_parquet(positions_path)
    required_columns = [
        "barcode", "in_tissue", "array_row", "array_col",
        "pxl_row_in_fullres", "pxl_col_in_fullres",
    ]
    missing_columns = sorted(set(required_columns) - set(pos.columns))
    if missing_columns:
        raise ValueError(f"Missing columns in positions parquet: {missing_columns}")
    if pos["barcode"].isna().any() or not pos["barcode"].is_unique:
        raise ValueError("Spatial position barcodes must be non-null and unique")

    pos = pos.set_index("barcode")
    missing_in_positions = adata.obs_names.difference(pos.index)
    extra_in_positions = pos.index.difference(adata.obs_names)
    print("Matrix barcodes:", adata.n_obs)
    print("Position barcodes:", len(pos))
    print("Missing in positions:", len(missing_in_positions))
    print("Extra in positions:", len(extra_in_positions))
    if len(missing_in_positions):
        raise ValueError(
            f"Missing coordinates for {len(missing_in_positions)} matrix barcodes"
        )

    aligned = pos.loc[adata.obs_names]
    for column in ("in_tissue", "array_row", "array_col"):
        adata.obs[column] = aligned[column].to_numpy()

    xy = aligned[
        ["pxl_col_in_fullres", "pxl_row_in_fullres"]
    ].to_numpy(dtype=np.float64)
    if not np.isfinite(xy).all():
        raise ValueError("Spatial coordinates contain NaN or infinite values")
    adata.obsm["spatial"] = xy
    return aligned, xy


aligned, xy = align_spatial_positions(adata, POSITIONS)
print("Spatial coordinates shape:", adata.obsm["spatial"].shape)
print("Tissue membership:", adata.obs["in_tissue"].value_counts().to_dict())
print("Fullres x range:", (xy[:, 0].min(), xy[:, 0].max()))
print("Fullres y range:", (xy[:, 1].min(), xy[:, 1].max()))
print(adata.obs.head())

## 3 · Inspect the supplied hires tissue image

The positions use **full-resolution pixels**, but the supplied PNG is a **hires image**. Since no `scalefactors_json.json` was provided, do **not** assume the two coordinate systems match. We record image dimensions only; no overlay or arbitrary rescaling is applied.

In [ ]:
def inspect_hires_image(image_path):
    """Inspect image dimensions without assuming fullres-to-hires scaling."""
    with Image.open(image_path) as img:
        return img.size[0], img.size[1], img.mode


image_width, image_height, image_mode = inspect_hires_image(IMAGE)
print("Hires image dimensions (width, height):", (image_width, image_height))
print("Image mode:", image_mode)
print("Fullres spatial coordinate bounds (x, y):",
      (xy[:, 0].min(), xy[:, 0].max()),
      (xy[:, 1].min(), xy[:, 1].max()))
print("Image-to-coordinate scale: NOT verified")

## 4 · Validate and save the AnnData object

Save the raw sparse matrix, gene metadata, bin metadata, and aligned full-resolution coordinates. The image itself remains an external file, referenced by a relative path.

In [ ]:
def validate_and_save_raw(adata, aligned, root, image_path, output_path,
                          prefix, image_width, image_height):
    """Validate the raw data and save it with spatial metadata."""
    assert adata.shape == (14400, 18085), f"Unexpected dataset shape: {adata.shape}"
    assert adata.obsm["spatial"].shape == (adata.n_obs, 2)
    assert adata.obs_names.equals(aligned.index), "Barcode order mismatch"
    assert sparse.issparse(adata.X), "Raw counts should remain sparse"

    adata.uns["dataset_id"] = prefix
    adata.uns["tissue"] = "human femoral head"
    adata.uns["spatial_coordinate_system"] = "fullres_pixels_xy"
    adata.uns["hires_image_relative_path"] = str(image_path.relative_to(root))
    adata.uns["hires_image_width"] = int(image_width)
    adata.uns["hires_image_height"] = int(image_height)
    adata.uns["image_alignment_verified"] = False

    output_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(output_path)
    return output_path


saved_path = validate_and_save_raw(
    adata, aligned, ROOT, IMAGE, OUTPUT, PREFIX, image_width, image_height
)
print("Saved:", saved_path)
print("File exists:", saved_path.is_file())
print("File size (MiB):", round(saved_path.stat().st_size / (1024**2), 2))

## 5 · Reload test (pipeline handoff)

Verify that the next notebook can read the saved artifact without relying on in-memory variables from this notebook.

In [ ]:
def verify_saved_anndata(output_path, original_adata):
    """Reload the saved file and verify the next notebook can use it."""
    check = sc.read_h5ad(output_path)
    assert check.shape == original_adata.shape
    assert check.obs_names.equals(original_adata.obs_names)
    assert check.var_names.equals(original_adata.var_names)
    assert np.array_equal(
        check.obsm["spatial"], original_adata.obsm["spatial"]
    )
    assert sparse.issparse(check.X)
    return check


check = verify_saved_anndata(OUTPUT, adata)
print("PASS — 01_data_loading completed")
print("AnnData:", check.shape)
print("Spatial coordinates:", check.obsm["spatial"].shape)
print("Spatial metadata:", check.obs.columns.tolist())
print("Output:", OUTPUT.relative_to(ROOT))

### Next: `02_quality_control.ipynb`

Input: `results/adata/01_raw.h5ad`. Compute QC metrics on raw counts and assess appropriate thresholds before filtering. **Important:** Image-to-coordinate registration is not yet verified; do not overlay bins onto the hires PNG until a trustworthy scale/transform is obtained.